# Transformer Embeddings

Evaluate a pretrained transformer as an alternative text
representation for human vs. AI classification.

We use frozen sentence embeddings and train a lightweight
classifier on the labeled training set.

We compare validation AUROC against our previous models and
investigate performance on unfamiliar writing styles.

In [48]:
from pathlib import Path
import sys
import torch
from sentence_transformers import SentenceTransformer

import pandas as pd

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from sklearn.feature_extraction.text import TfidfVectorizer

In [49]:
# locate 
project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

sys.path.insert(0, str(project_dir))

from data_utils import load_data, split_data

# check environment
device = "mps" if torch.backends.mps.is_available() else "cpu"

print("Project directory:", project_dir)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", device)


Project directory: /Users/constancechen/Downloads/human-or-ai-submissions/submissions/group-38
Python: 3.13.9
PyTorch: 2.15.0
Device: mps


In [50]:
zip_path = (project_dir / "data" / "cornell-tech-cs-5785-combined-2026-ai-vs-human-text.zip")

train, unlabeled, test = load_data(zip_path)

# same 80/20 split used in previous notebooks
train_df, val_df = split_data(train)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Kaggle test:", test.shape)

assert len(train_df) == 8000
assert len(val_df) == 2000
assert set(train_df["id"]).isdisjoint(set(val_df["id"]))

Training: (8000, 3)
Validation: (2000, 3)
Kaggle test: (3000, 2)


In [51]:
# load pretrained transformer encoder
model_name = "sentence-transformers/all-MiniLM-L6-v2"

encoder = SentenceTransformer(model_name, device=device)

print("Model:", model_name)
print("Device:", device)
print("Embedding dimension:", encoder.get_sentence_embedding_dimension())
print("Maximum sequence length:", encoder.max_seq_length)

# test the encoder on a few passages
sample_texts = train_df["text"].iloc[:3].tolist()

sample_embeddings = encoder.encode(
    sample_texts,
    batch_size=3,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Sample embedding shape:", sample_embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: sentence-transformers/all-MiniLM-L6-v2
Device: mps
Embedding dimension: 384
Maximum sequence length: 256
Sample embedding shape: (3, 384)


/var/folders/01/vrhtdbr91m3429sptw2x6s3h0000gn/T/ipykernel_27567/2930156766.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", encoder.get_sentence_embedding_dimension())


In [52]:
# prepare text inputs
train_texts = train_df["text"].fillna("").astype(str).tolist()
val_texts = val_df["text"].fillna("").astype(str).tolist()

# generate embeddings using the frozen transformer
X_train_emb = encoder.encode(
    train_texts,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

X_val_emb = encoder.encode(
    val_texts,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# verify embedding dimensions
assert X_train_emb.shape == (8000, 384)
assert X_val_emb.shape == (2000, 384)
assert np.isfinite(X_train_emb).all()
assert np.isfinite(X_val_emb).all()

print("Training embeddings:", X_train_emb.shape)
print("Validation embeddings:", X_val_emb.shape)


Batches:   0%|          | 0/500 [00:00<?, ?it/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Training embeddings: (8000, 384)
Validation embeddings: (2000, 384)


In [53]:
# get the training and validation labels
y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()

# train a classifier using the frozen transformer embeddings
embedding_lr = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42
)

embedding_lr.fit(X_train_emb, y_train)

# predict probabilities on the validation set
val_probs = embedding_lr.predict_proba(X_val_emb)[:, 1]

# evaluate validation AUROC
val_auc = roc_auc_score(y_val, val_probs)

print(f"Transformer embedding LR AUROC: {val_auc:.6f}")

Transformer embedding LR AUROC: 0.782262


In [54]:
# load our previous validation predictions
hybrid_val = pd.read_csv(project_dir / "hybrid_val_predictions.csv")

mia_val = pd.read_csv(
    project_dir / "improved_ensemble" / "results" / "improved_validation.csv"
)

# attach MiniLM predictions to their passage IDs
minilm_val = pd.DataFrame({
    "id": val_df["id"].to_numpy(),
    "minilm_prob": val_probs
})

# align all three models by ID
comparison = hybrid_val.merge(
    mia_val[["id", "label", "selected_probability_ai"]],
    on="id",
    suffixes=("_hybrid", "_mia"),
    validate="one_to_one"
).merge(
    minilm_val,
    on="id",
    validate="one_to_one"
)

assert len(comparison) == 2000
assert (comparison["label_hybrid"] == comparison["label_mia"]).all()

y_true = comparison["label_hybrid"]

# original 25/75 blend
original_probs = (
    0.25 * comparison["hybrid_prob"]
    + 0.75 * comparison["selected_probability_ai"]
)

# add a small 10% contribution from MiniLM
new_probs = (
    0.90 * original_probs
    + 0.10 * comparison["minilm_prob"]
)

original_auc = roc_auc_score(y_true, original_probs)
new_auc = roc_auc_score(y_true, new_probs)

print(f"Original blend AUROC: {original_auc:.6f}")
print(f"Blend + MiniLM AUROC: {new_auc:.6f}")
print(f"Difference:          {new_auc - original_auc:+.6f}")


Original blend AUROC: 0.998052
Blend + MiniLM AUROC: 0.998244
Difference:          +0.000192


In [55]:
# train a domain classifier using training and Kaggle test texts
domain_texts = pd.concat([
    train_df["text"],
    test["text"]
]).fillna("")

domain_labels = np.concatenate([
    np.zeros(len(train_df)),
    np.ones(len(test))
])

domain_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=30000,
    sublinear_tf=True
)

X_domain = domain_vec.fit_transform(domain_texts)

domain_clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
domain_clf.fit(X_domain, domain_labels)

# measure how similar validation examples are to Kaggle test
val_domain_features = domain_vec.transform(
    val_df["text"].fillna("")
)

test_likeness = pd.DataFrame({
    "id": val_df["id"].to_numpy(),
    "score": domain_clf.predict_proba(val_domain_features)[:, 1]
})

# attach test-likeness scores to our model predictions
analysis = comparison.merge(
    test_likeness,
    on="id",
    validate="one_to_one"
)

# focus on the 25% most test-like validation examples
hard = analysis[
    analysis["score"] >= analysis["score"].quantile(0.75)
]

y_hard = hard["label_hybrid"]

original_hard = (
    0.25 * hard["hybrid_prob"]
    + 0.75 * hard["selected_probability_ai"]
)

minilm_hard = (
    0.90 * original_hard
    + 0.10 * hard["minilm_prob"]
)

print("Most test-like examples:", len(hard))
print(f"Original blend AUROC: {roc_auc_score(y_hard, original_hard):.6f}")
print(f"Blend + MiniLM AUROC: {roc_auc_score(y_hard, minilm_hard):.6f}")
print(f"Difference: {roc_auc_score(y_hard, minilm_hard) - roc_auc_score(y_hard, original_hard):+.6f}")


Most test-like examples: 500
Original blend AUROC: 0.994576
Blend + MiniLM AUROC: 0.995755
Difference: +0.001179


In [56]:
# save MiniLM validation predictions for future blending
minilm_val_df = pd.DataFrame({
    "id": val_df["id"].to_numpy(),
    "label": y_val,
    "minilm_prob": val_probs
})

assert len(minilm_val_df) == 2000
assert minilm_val_df["id"].is_unique
assert minilm_val_df["minilm_prob"].between(0, 1).all()

# keep generated predictions local
output_path = project_dir / "data" / "minilm_val_predictions.csv"
minilm_val_df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", minilm_val_df.shape)
print(minilm_val_df.head())


Saved to: /Users/constancechen/Downloads/human-or-ai-submissions/submissions/group-38/data/minilm_val_predictions.csv
Shape: (2000, 3)
     id  label  minilm_prob
0  5865      0     0.530169
1  1774      1     0.330316
2  3336      0     0.641426
3  9082      1     0.723248
4  6970      0     0.354062


In [57]:
# generate MiniLM embeddings for Kaggle test passages
test_texts = test["text"].fillna("").astype(str).tolist()

X_test_emb = encoder.encode(
    test_texts,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# verify test embeddings
assert X_test_emb.shape == (3000, 384)
assert np.isfinite(X_test_emb).all()

print("Test embeddings:", X_test_emb.shape)


Batches:   0%|          | 0/188 [00:00<?, ?it/s]

Test embeddings: (3000, 384)


In [58]:
# combine training and validation embeddings for final training
X_full_emb = np.vstack([X_train_emb, X_val_emb])
y_full = np.concatenate([y_train, y_val])

assert X_full_emb.shape == (10000, 384)
assert len(y_full) == 10000

# train final classifier using all labeled examples
final_embedding_lr = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42
)

final_embedding_lr.fit(X_full_emb, y_full)

# generate Kaggle test probabilities
test_minilm_probs = final_embedding_lr.predict_proba(X_test_emb)[:, 1]

# verify predictions
assert len(test_minilm_probs) == 3000
assert np.isfinite(test_minilm_probs).all()
assert np.all((test_minilm_probs >= 0) & (test_minilm_probs <= 1))

print("Full training embeddings:", X_full_emb.shape)
print("Test predictions:", test_minilm_probs.shape)
print("First five probabilities:", test_minilm_probs[:5])


Full training embeddings: (10000, 384)
Test predictions: (3000,)
First five probabilities: [0.55435062 0.45897211 0.51695004 0.27067458 0.55210987]


In [59]:
# save MiniLM test predictions for blending
minilm_test_df = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "minilm_prob": test_minilm_probs
})

# verify predictions align with Kaggle test examples
assert len(minilm_test_df) == 3000
assert minilm_test_df["id"].is_unique
assert minilm_test_df["minilm_prob"].between(0, 1).all()

# save locally; do not commit generated prediction CSVs
output_path = project_dir / "data" / "minilm_test_predictions.csv"
minilm_test_df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", minilm_test_df.shape)
print(minilm_test_df.head())


Saved to: /Users/constancechen/Downloads/human-or-ai-submissions/submissions/group-38/data/minilm_test_predictions.csv
Shape: (3000, 2)
      id  minilm_prob
0  13001     0.554351
1  13002     0.458972
2  13003     0.516950
3  13004     0.270675
4  13005     0.552110


In [60]:
# load previously generated Kaggle predictions
hybrid_test = pd.read_csv(project_dir / "submission_hybrid_mlp.csv")

# use Mia's original Kaggle-scored submission
mia_test = pd.read_csv(
    project_dir / "data" / "submission_improved.csv"
)

minilm_test = pd.read_csv(
    project_dir / "data" / "minilm_test_predictions.csv"
)

# align all three models by passage ID
test_comparison = (
    hybrid_test.rename(columns={"label": "hybrid_prob"})
    .merge(
        mia_test.rename(columns={"label": "mia_prob"}),
        on="id",
        validate="one_to_one"
    )
    .merge(minilm_test, on="id", validate="one_to_one")
)

# verify all test examples are present
assert len(test_comparison) == 3000
assert test_comparison["id"].is_unique
assert set(test_comparison["id"]) == set(test["id"])

# weighted average of the three models
test_comparison["label"] = (
    0.225 * test_comparison["hybrid_prob"]
    + 0.675 * test_comparison["mia_prob"]
    + 0.10 * test_comparison["minilm_prob"]
)

# preserve original Kaggle test ID order
blend_submission = (
    test[["id"]]
    .merge(
        test_comparison[["id", "label"]],
        on="id",
        how="left",
        validate="one_to_one"
    )
)

assert blend_submission.shape == (3000, 2)
assert blend_submission["label"].notna().all()
assert blend_submission["label"].between(0, 1).all()

# save locally; do not commit generated CSVs
output_path = project_dir / "submission_blend_minilm_10.csv"
blend_submission.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", blend_submission.shape)
print(blend_submission.head())


Saved to: /Users/constancechen/Downloads/human-or-ai-submissions/submissions/group-38/submission_blend_minilm_10.csv
Shape: (3000, 2)
      id     label
0  13001  0.953300
1  13002  0.083427
2  13003  0.126800
3  13004  0.056258
4  13005  0.946957


## Results and Discussion

We evaluated a frozen pretrained MiniLM encoder with Logistic Regression for human-vs-AI classification.

| Model | Validation AUROC |
|---|---:|
| MiniLM + Logistic Regression | 0.782262 |
| Previous 25/75 ensemble | 0.998052 |
| Ensemble + 10% MiniLM | 0.998244 |

Although MiniLM performed poorly as a standalone classifier, adding a 10% contribution improved ensemble validation AUROC.

On the 500 most test-like validation examples, the ensemble improved from 0.994576 to 0.995755.

The three-model blend achieved a public Kaggle AUROC of **0.96530**, compared with 0.96436 for the previous ensemble.

These improvements are exploratory because the same validation split was used for multiple model-selection experiments. Public leaderboard performance may not reflect private leaderboard performance.

**Pretrained model:** `sentence-transformers/all-MiniLM-L6-v2` (Apache 2.0). No external labeled datasets were used.
